# roundtrip-closure — Colab Pilot Run

**PhD Thesis Chapter 3:** Heterogeneous Multi-SLM Closure of the Docstring–Test–Code Triangle: A Mutation-Testing Study.

This notebook runs the **30-function × 6-cell pilot** end-to-end on Colab Pro+ A100.

## Before you start

1. **Runtime → Change runtime type → A100 GPU** (or T4 if A100 unavailable; pilot will run slower).
2. **Runtime → Run all** does the whole thing once you're set up. Or step through cell-by-cell.

## Expected wall-clock

| Phase | First session | Subsequent sessions |
|---|---|---|
| Mount Drive + clone repo + install deps | 2 min | 1 min |
| Install Ollama + start server | 1 min | 1 min |
| Pull 7 SLMs (~83 GB) | **30–60 min** | 0 (cached on Drive) |
| `prepare_roundtrip.py` (datasets + decontamination) | 5–10 min | 0 (cached) |
| Pilot: 30 functions × 6 cells × 3 paths = 540 round-trips | **30–60 min** | resume from checkpoint |

## Disconnect resilience

Everything is stored on **Google Drive** via symlinks:

| Mounted location | Purpose |
|---|---|
| `Drive/roundtrip-closure-state/ollama_models/` | Pulled SLM weights (~83 GB) |
| `Drive/roundtrip-closure-state/data/` | HumanEval + MBPP + LiveCodeBench + HumanEval-Mutated JSONL |
| `Drive/roundtrip-closure-state/checkpoints/` | LLM-call cache (SHA-256 keyed) |
| `Drive/roundtrip-closure-state/results/` | TSV results + pilot summary |
| `Drive/roundtrip-closure-state/logs/` | Per-cell logs |

If Colab disconnects mid-run, just **Runtime → Run all** in a new session — every completed `(cell, sample, path)` tuple resumes from the TSV; every cached LLM call is a free Drive lookup.

## 1. GPU sanity check

In [ ]:
!nvidia-smi

## 2. Mount Google Drive (persistent storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Set up Drive layout + symlink Ollama models

This is what makes the run **idempotent across sessions**. The symlinks redirect:
- Ollama's `/root/.ollama/models/` → Drive  (so model pulls happen once, ever)
- Repo's `data/ checkpoints/ results/ logs/` → Drive  (so cache + outputs survive)

In [ ]:
import os
import shutil
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/roundtrip-closure-state')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# Persistent subdirs on Drive
for subdir in ('data', 'checkpoints', 'results', 'logs', 'ollama_models'):
    (DRIVE_ROOT / subdir).mkdir(exist_ok=True)

# Redirect Ollama's local model storage to Drive
os.makedirs('/root/.ollama', exist_ok=True)
ollama_models_link = Path('/root/.ollama/models')
ollama_models_target = DRIVE_ROOT / 'ollama_models'

if ollama_models_link.is_symlink():
    ollama_models_link.unlink()
elif ollama_models_link.exists():
    shutil.rmtree(ollama_models_link)

ollama_models_link.symlink_to(ollama_models_target)

print(f'  Drive state root  : {DRIVE_ROOT}')
print(f'  Ollama models →   : {ollama_models_link} -> {os.readlink(ollama_models_link)}')

# Show what's already pulled (zero on first session, full lineup on later sessions)
already_pulled = list((DRIVE_ROOT / 'ollama_models' / 'manifests' / 'registry.ollama.ai').rglob('*'))
print(f'  Models already on Drive: {len(already_pulled)} manifest files')

## 4. Clone (or pull) the repo

In [ ]:
import os
REPO_URL  = 'https://github.com/balajivenky06/roundtrip-closure'
REPO_PATH = '/content/roundtrip-closure'

if os.path.exists(REPO_PATH):
    !cd {REPO_PATH} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_PATH}

os.chdir(REPO_PATH)
print(f'Working dir: {os.getcwd()}')
!git log --oneline -5

## 5. Symlink repo's data/cache/results/logs to Drive

After this cell, every read/write the code does to `data/`, `checkpoints/`, `results/`, or `logs/` actually goes to Drive — and survives a Colab disconnect.

In [ ]:
import shutil
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/roundtrip-closure-state')
REPO_PATH  = Path('/content/roundtrip-closure')

for subdir in ('data', 'checkpoints', 'results', 'logs'):
    drive_target = DRIVE_ROOT / subdir
    repo_dir = REPO_PATH / subdir

    # If repo has a real (non-symlink) dir, move its contents to Drive then symlink
    if repo_dir.exists() and not repo_dir.is_symlink():
        for item in list(repo_dir.iterdir()):
            dest = drive_target / item.name
            if dest.exists():
                continue  # Drive copy wins on conflict
            shutil.move(str(item), str(dest))
        try:
            repo_dir.rmdir()
        except OSError:
            shutil.rmtree(repo_dir)

    if repo_dir.is_symlink():
        repo_dir.unlink()

    repo_dir.symlink_to(drive_target)
    print(f'  {repo_dir} → {drive_target}')

## 6. Install Python dependencies

Uses the project's `pyproject.toml` — ollama, datasets, bert-score, statsmodels, etc.

In [ ]:
!pip install -q -e .
print('Done.')

## 7. Install Ollama

The Ollama daemon isn't pre-installed on Colab. This cell installs it via the official script.

In [ ]:
import shutil
if shutil.which('ollama'):
    print(f'Ollama already installed at: {shutil.which("ollama")}')
else:
    !curl -fsSL https://ollama.com/install.sh | sh
    print('Ollama installed.')

!ollama --version

## 8. Start the Ollama server in the background

Daemon must be running for `ollama pull` and `ollama_client.call_llm` to work. We start it as a detached subprocess so the notebook can keep going.

In [ ]:
import subprocess
import time
import requests

def is_ollama_up(timeout: float = 2.0) -> bool:
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=timeout)
        return r.status_code == 200
    except Exception:
        return False

if is_ollama_up():
    print('Ollama server already running.')
else:
    # Start detached
    subprocess.Popen(
        ['ollama', 'serve'],
        stdout=open('/tmp/ollama.log', 'w'),
        stderr=subprocess.STDOUT,
    )
    for _ in range(30):
        if is_ollama_up():
            print('Ollama server up.')
            break
        time.sleep(1)
    else:
        raise RuntimeError('Ollama server failed to start within 30 s. Check /tmp/ollama.log')

!ollama list

## 9. Pull all 7 SLMs (skip already-cached)

Total ~83 GB. First session: 30–60 min over Colab's network. Subsequent sessions: instant (everything is on Drive via the symlink set up earlier).

If a model already exists, `ollama pull` is essentially a no-op.

In [ ]:
MODELS = [
    'llama3.2:3b',              # ~2 GB — small floor (Meta)
    'phi4:14b',                  # ~9 GB — mid dense (Microsoft)
    'qwen3.6:27b',               # ~16 GB — latest dense (Alibaba, April 2026)
    'gemma4:26b',                # ~16 GB — latest MoE (Google, March 2026)
    'mistral-small3.2:24b',      # ~14 GB — latest Mistral
    'qwen3-coder:30b',           # ~17 GB — coder specialist (Alibaba)
    'deepseek-r1:14b',           # ~9 GB — judge LLM (DeepSeek)
]

for tag in MODELS:
    print(f'\n=== {tag} ===')
    !ollama pull {tag}

print('\nAll models pulled. Final list:')
!ollama list

## 10. One-time dataset prep

Downloads HumanEval + MBPP, builds:
- `data/core_sample_150.jsonl`  (150 seed-42 stratified samples)
- `data/livecodebench_25.jsonl`  (post-2024-12-01 subset; skipped if unavailable)
- `data/humaneval_mutated_50.jsonl`  (decontaminated subset via `decontaminate.py`)

Since `data/` is symlinked to Drive, this runs once ever.

In [ ]:
!python3 prepare_roundtrip.py

## 11. Run the pilot

30 functions × 6 cells × 3 paths = 540 round-trips.

Writes to `results/pilot_results.tsv` (on Drive) and `results/pilot_summary.md`.

If Colab disconnects, re-running this cell resumes from the last completed `(cell, sample, path)` tuple — every cached LLM call is a free Drive lookup.

In [ ]:
!python3 scripts/run_pilot.py

## 12. View pilot summary

In [ ]:
from IPython.display import Markdown, display
from pathlib import Path

summary_path = Path('results/pilot_summary.md')
if summary_path.exists():
    display(Markdown(summary_path.read_text()))
else:
    print('No pilot_summary.md yet; re-run the cell above.')

## 13. Inspect raw results

In [ ]:
import pandas as pd
from pathlib import Path

tsv = Path('results/pilot_results.tsv')
if tsv.exists() and tsv.stat().st_size > 0:
    df = pd.read_csv(tsv, sep='\t')
    print(f'Total rows: {len(df)}')
    print('\nPer-cell row counts:')
    print(df.groupby(['cell_id', 'path']).size().unstack(fill_value=0))
    print('\nMean metric_value by (cell, path):')
    print(df.groupby(['cell_id', 'path'])['metric_value'].mean().unstack(fill_value=float('nan')))
    print('\nJudge rating distribution:')
    print(df['judge_rating'].value_counts().sort_index())
else:
    print('No pilot_results.tsv yet.')

## 14. Run the analysis pipeline on pilot results

Produces:
- `tables/*.tex` + `tables/*.csv` — 11 LaTeX + CSV tables ready to paste into `paper_draft.tex`
- `plots/output/*.png` — 9 publication-quality figures
- `results/analysis_summary.json` — every statistical-test output (machine-readable)
- `results/paper_ready_summary.md` — copy-paste-ready numbers
- `results/human_eval_worksheet_60.csv` — blinded annotation worksheet for the human-eval study

Statistical tests executed: Type-III ANOVA, Tukey HSD (190-pair Bonferroni), per-stage bottleneck decomposition, cross-cell Spearman ρ, judge↔metric Pearson r, false-closure rate, contamination sensitivity (Welch t), cache efficiency.

In [ ]:
!python3 scripts/run_analysis.py --pilot

## 15. View the paper-ready summary

In [ ]:
from IPython.display import Markdown, display
from pathlib import Path

summary_path = Path('results/paper_ready_summary.md')
if summary_path.exists():
    display(Markdown(summary_path.read_text()))
else:
    print('No paper_ready_summary.md yet; re-run the analysis cell above.')

## 16. Browse all 9 generated figures inline

In [ ]:
from IPython.display import Image, display, Markdown
from pathlib import Path

fig_dir = Path('plots/output')
for fig in sorted(fig_dir.glob('fig*.png')):
    display(Markdown(f'### {fig.stem}'))
    display(Image(filename=str(fig), width=820))

## 17. List generated tables

Each table is in both `.tex` (paste into the manuscript) and `.csv` (open in Excel) formats.

In [ ]:
from pathlib import Path

tables_dir = Path('tables')
print(f'{len(list(tables_dir.glob("*.tex")))} LaTeX tables + matching CSVs:')
for tex in sorted(tables_dir.glob('*.tex')):
    print(f'  {tex.relative_to(Path("."))}')

# Sneak preview: print the closure-rate matrix CSV inline
import pandas as pd
csv_path = tables_dir / 'tab_closure_rate_matrix.csv'
if csv_path.exists():
    print('\n--- tab_closure_rate_matrix preview ---')
    print(pd.read_csv(csv_path).to_string(index=False))

## 18. Full sweep (after the pilot passes)

If the pilot verdict is **GO** or **GO_WITH_NOTES**, run all 20 cells of the full DOE.

**Estimated wall-clock:** ~24–36 hours of A100 time across 20 cells × 150 functions × 3 paths. Resume-safe across Colab disconnects.

Each cell runs independently in its own subprocess — if one cell crashes, the others continue. Per-cell results land in `results/results_roundtrip.tsv` (different file than the pilot's TSV).

In [ ]:
# Run a single full-sweep cell. Repeat for each of the 20 cells.
# Set CELL_ID environment variable, then run train_roundtrip.py.

import os, subprocess

CELL = 'M1'                     # one of: M1-M6, H1-H11, N1-N3
DATASET = 'core'                # core | livecodebench | humaneval_mutated

env = {**os.environ,
       'CELL_ID': CELL,
       'DATASET': DATASET,
       'N_SAMPLES': '150'}

subprocess.run(['python3', 'train_roundtrip.py'], env=env, check=False)

In [ ]:
# OR — run all 20 cells back-to-back.
# Comment out cells you've already finished if you want to skip them.

import os, subprocess, time
from datetime import datetime

ALL_CELLS = [
    'M1','M2','M3','M4','M5','M6',
    'H1','H2','H3','H4','H5','H6','H7','H8','H9','H10','H11',
    'N1','N2','N3',
]

for cell in ALL_CELLS:
    print(f'\n{"═"*60}\n  {datetime.now().strftime("%H:%M:%S")} — cell {cell}\n{"═"*60}')
    env = {**os.environ, 'CELL_ID': cell, 'DATASET': 'core', 'N_SAMPLES': '150'}
    ret = subprocess.run(['python3', 'train_roundtrip.py'], env=env, check=False)
    print(f'  exit={ret.returncode}')
    time.sleep(2)

## Disconnect recovery cheat-sheet

**If Colab disconnects mid-sweep:**
1. Reconnect — Runtime → Reconnect (or open the notebook in a new tab).
2. Re-run cells 1–8 (mount drive, clone, install, start Ollama). All are idempotent.
3. Skip cell 9 — models are already on Drive.
4. Skip cell 10 — datasets are already on Drive.
5. Re-run cell 11 (pilot) or cells 14 (full sweep). The driver will load completed `(cell, sample, path)` keys from the TSV and skip them.

**If you suspect cache corruption** (very rare):
```python
import closure_cache
closure_cache.clear()    # nuclear option — wipes the entire cache
```

**To inspect cache stats:**
```python
import closure_cache
print(closure_cache.stats())
```

## Going further

After all 20 cells complete:
- `results/results_roundtrip.tsv` has the full sweep — ~9,000 rows for the core sweep, ~13,500 with held-out subsets.
- Run the analysis script (to be added in a later batch) to produce the mixed-effects regression + Tukey + per-stage decomposition tables.
- Sample the 60 pairs for the human-evaluation study per the §4.6 sampling strategy.

---

Repository: https://github.com/balajivenky06/roundtrip-closure